# 08 — Demand Forecast (Arrival Volume)

Predicts **how many customers will arrive** per branch over the next 7 days, respecting each
branch's configured schedule and the public-holiday calendar. Honest by construction: a
GradientBoosting model is backtested against a seasonal-naive baseline and we use whichever wins.

**Outputs the `demand_forecast` insight** consumed by the Manager & Executive dashboards and the staffing model.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams.update({"axes.titleweight": "bold", "axes.titlesize": 12, "figure.dpi": 110})
pd.set_option("display.max_columns", 40)

# Lyne palette (matches the admin dashboards)
NAVY, STEEL, TEAL, RED, GOLD = "#2F5063", "#6E8AA6", "#2E7387", "#B23A4E", "#9A6B2E"
BLUES = sns.light_palette(NAVY, n_colors=6, reverse=True)

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
DOW = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
def hour_label(h): return f"{((int(h) + 11) % 12) + 1}{'am' if h < 12 else 'pm'}"
print("Ready.")

In [ ]:
from scripts import forecast_demand as fd

conn = fd.connect()
df = fd.load_arrivals(conn)
holidays = fd.load_holidays(conn)
branch_cal = fd.load_branch_calendar(conn)
print(f"{len(df):,} (branch, service, day, hour) arrival rows · "
      f"{df.visit_date.min().date()} → {df.visit_date.max().date()}")
df.head()

## 2. Backtest — does the model beat a seasonal-naive baseline?\nTrained on all but the last 14 days, scored on the held-out window.

In [ ]:
bt = fd.backtest(df, holidays)
bt_df = pd.DataFrame([
    {"model": "GradientBoosting", "MAE (arrivals/hr)": bt["gbr_mae"]},
    {"model": "Seasonal-naive",   "MAE (arrivals/hr)": bt["seasonal_naive_mae"]},
])
print(f"Holdout: {bt['holdout_days']} days, {bt['test_rows']:,} rows · winner → {bt['chosen']}")

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.barh(bt_df["model"], bt_df["MAE (arrivals/hr)"], color=[NAVY, STEEL])
ax.set_title("Backtest error — lower is better"); ax.set_xlabel("MAE (arrivals / hour)")
for b, v in zip(bars, bt_df["MAE (arrivals/hr)"]):
    ax.text(v, b.get_y() + b.get_height()/2, f" {v:.3f}", va="center", fontweight="bold")
ax.invert_yaxis(); plt.tight_layout(); plt.show()
bt_df

## 3. Next 7 days — expected arrivals per branch

In [ ]:
fut = fd.forecast(df, holidays, branch_cal)
daily = (fut.groupby([fut.branch_name, fut.visit_date.dt.date])["predicted_arrivals"].sum()
            .round().astype(int).reset_index())
daily.columns = ["branch", "date", "expected_arrivals"]
pivot = daily.pivot(index="branch", columns="date", values="expected_arrivals").fillna(0).astype(int)
pivot

In [ ]:
ax = pivot.T.plot(kind="bar", figsize=(12, 5), width=0.8,
                  color=sns.color_palette("Blues_r", n_colors=len(pivot)))
ax.set_title("Expected Arrivals — Next 7 Days by Branch")
ax.set_ylabel("Expected arrivals"); ax.set_xlabel("")
ax.legend(title="Branch", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 4. Typical intraday shape (all branches)

In [ ]:
shape = fut.groupby("hour")["predicted_arrivals"].mean().reset_index()
fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(shape["hour"], shape["predicted_arrivals"], color=NAVY, alpha=0.12)
ax.plot(shape["hour"], shape["predicted_arrivals"], color=NAVY, lw=2.6, marker="o")
ax.set_title("Average Predicted Arrivals by Hour"); ax.set_xlabel("Hour of day"); ax.set_ylabel("Arrivals / hour")
ax.set_xticks(shape["hour"]); ax.set_xticklabels([hour_label(h) for h in shape["hour"]])
plt.tight_layout(); plt.show()

## 5. Persist (optional)\nSet `WRITE_DB=1` before launching to upsert the `demand_forecast` insight into `predictive_results`.

In [ ]:
if os.getenv("WRITE_DB") == "1":
    ins, gen, stale = fd.build_insights(df, fut, bt)
    fd.upsert_insights(conn, ins, gen, stale, fd.MODEL_VERSION)
    print(f"Upserted {len(ins)} demand_forecast insight(s).")
else:
    print("Preview only — set WRITE_DB=1 to persist.")
conn.close()